# NLP Prototyping for Public Health Response (Google Colab)
Ensure your Colab environment has a CPU or T4 GPU attached. Run the cells sequentially.

### Cell 1: Setup and Installations
Install the required Hugging Face libraries if they aren't already present.

In [6]:
# Setup and Installations (Now includes sentencepiece and protobuf)
!pip install -q sentencepiece protobuf

### Cell 2: Imports and Data Loading Configuration
We fetch the processed CSVs directly from the main branch of the GitHub repository.

In [1]:
import pandas as pd
from transformers import pipeline

# Pointing directly to the 'processed' folder on the 'main' branch of the INRB repo
REPO_BASE_URL = "https://raw.githubusercontent.com/INRB-UMIE/BDBV2026-Data/main"
BASE_CSV_URL = f"{REPO_BASE_URL}/data/public_health_response/processed"

def load_data(filename):
    """Helper function to load CSV from the raw GitHub URL."""
    url = f"{BASE_CSV_URL}/{filename}"
    print(f"Downloading: {url}")
    df = pd.read_csv(url)

    # Filter out completely empty narratives if any exist
    if df.shape[1] > 2:
        text_col = df.columns[2]
        df = df.dropna(subset=[text_col])
    return df

### Cell 3: Task 1 - Named Entity Recognition (NER)
**Goal:** Automatically extract Locations, Organizations, and Persons from the `management` pillar.

In [2]:
print("\n--- Task 1: Named Entity Recognition ---")
df_management = load_data("public_health_response__epidemiological_management_en__daily.csv")
text_col_man = "management_en"

# Initialize NER pipeline
ner_pipeline = pipeline("ner", model="dslim/bert-base-NER", aggregation_strategy="simple", device=-1)

# Preview on the first 3 rows
sample_texts_ner = df_management[text_col_man].head(3).tolist()

for i, text in enumerate(sample_texts_ner):
    print(f"\n[Original Text {i+1}]: {text}")
    entities = ner_pipeline(text)
    print("-> Extracted Entities:")
    for entity in entities:
        print(f"   - {entity['entity_group']}: {entity['word']} (Confidence: {entity['score']:.2f})")


--- Task 1: Named Entity Recognition ---
Downloading: https://raw.githubusercontent.com/INRB-UMIE/BDBV2026-Data/main/data/public_health_response/processed/public_health_response__epidemiological_management_en__daily.csv


config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  433MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]


[Original Text 1]: Patients in isolation: HGR 19, SOFEPADI 4, Clinique Bénédicte 2, RWAOLE 9, CH Elikya 6 (total 40). New admissions: 12 (0 PPL, 12 others). Deaths: 3 (0 confirmed, 3 suspects). Escapes: 1.
-> Extracted Entities:
   - ORG: HGR (Confidence: 0.95)
   - ORG: S (Confidence: 0.99)
   - ORG: ##OFEPADI (Confidence: 0.95)
   - ORG: Clinique Bénédicte (Confidence: 0.98)
   - ORG: RWAOLE (Confidence: 0.98)
   - ORG: CH Elikya (Confidence: 0.69)

[Original Text 2]: Patients in isolation: HGR 24, CME 8, CH Salama 14, CS Hoho 8 (total 54). New admissions: 17 (2 PPL, 15 others). Deaths: 2 (0 confirmed, 2 suspects). Non-cases discharged: 13. Escapes: 1. Transferred to HGR: 1.
-> Extracted Entities:
   - ORG: HGR (Confidence: 0.76)
   - ORG: CME (Confidence: 0.94)
   - PER: CH Salama (Confidence: 1.00)
   - PER: CS Hoho (Confidence: 0.95)

[Original Text 3]: Patients in isolation: 30. New admissions: 9 (5 PPL, 4 others). Deaths: 1 (0 confirmed, 1 suspect). Escapes: 2.
-> Extracted Ent

### Cell 4: Task 2 - Text Summarization
**Goal:** Compress multiple daily `logistics` updates into a short executive summary.

In [11]:
df_logistics

,nom,date,logistics_en
0,Bunia,30-05-2026,Delivery of 5 tons of medicines (support UG-PD...
1,Rwampara,31-05-2026,Inauguration and commissioning of the standard...
2,Bunia,02-06-2026,Official handover of 3 vehicles (government do...
3,Rwampara,02-06-2026,Official handover of 3 vehicles (government do...
4,Mongbalu,02-06-2026,Official handover of 3 vehicles (government do...
...,...,...,...
56,Logo,22-06-2026,Delivery of IPC inputs and medicines.
57,Aru,22-06-2026,Delivery of IPC inputs and medicines.
58,Mongbwalu,22-06-2026,Delivery of IPC inputs and medicines.
59,Lita,23-06-2026,Delivery of UNICEF medicine supplies.


In [10]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

print("\n--- Task 2: Text Summarization ---")
df_logistics = load_data("public_health_response__epidemiological_logistics_en__daily.csv")
text_col_log = "logistics_en"
combined_logistics_text = " ".join(df_logistics[text_col_log].head(5).tolist())

# 1. Load the model and tokenizer explicitly
model_name = "sshleifer/distilbart-cnn-12-6"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print(f"\n[Combined Logistics Input (Length: {len(combined_logistics_text)} chars)]:\n{combined_logistics_text}\n")

# 2. Tokenize the input (truncating if it exceeds the model's 1024 token limit)
inputs = tokenizer(combined_logistics_text, return_tensors="pt", max_length=1024, truncation=True)

# 3. Generate the summary
summary_ids = model.generate(
    inputs["input_ids"],
    max_length=60,
    min_length=20,
    do_sample=False
)

# 4. Decode the output back into text
summary_text = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print("-> Executive Summary:")
print(summary_text)


--- Task 2: Text Summarization ---


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.22GB            

...

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.22GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/358 [00:00<?, ?it/s]


[Combined Logistics Input (Length: 850 chars)]:
Delivery of 5 tons of medicines (support UG-PDSS). Delivery of demarcation nets to Bunia CME. Inauguration and commissioning of the standardised ETC at CME Rwampara, newly rehabilitated. Official handover of 3 vehicles (government donation) by the Governor of Ituri province to CDPS Ituri. Two vehicles immediately assigned to Rwampara and Nyankunde health zones. Official handover of 3 vehicles (government donation) by the Governor of Ituri province to CDPS Ituri. Two vehicles immediately assigned to Rwampara and Nyankunde health zones. Delivery of medicines and IPC inputs with support from UG-PDSS. Official handover of 3 vehicles (government donation) by the Governor of Ituri province to CDPS Ituri. Two vehicles immediately assigned to Rwampara and Nyankunde health zones. Delivery of IPC prevention and control inputs to these health zones.

-> Executive Summary:
 Delivery of 5 tons of medicines (support UG-PDSS). Delivery of demarcation n

In [13]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

print("--- Task 2: Daily Events Summarization ---")

# 1. Load the dataset directly from the URL
url = "https://raw.githubusercontent.com/INRB-UMIE/BDBV2026-Data/main/data/public_health_response/processed/public_health_response__epidemiological_logistics_en__daily.csv"
df_logistics = pd.read_csv(url)

# 2. Convert dates to standard format and sort chronologically
df_logistics['date_parsed'] = pd.to_datetime(df_logistics['date'], format='%d-%m-%Y', errors='coerce')
df_logistics = df_logistics.sort_values('date_parsed')

# 3. Load the model and tokenizer explicitly
model_name = "sshleifer/distilbart-cnn-12-6"
print(f"Loading model: {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
print("Model loaded successfully.\n")

# 4. Group by date and summarize
# sort=False ensures it respects the chronological sorting we just did
grouped = df_logistics.groupby('date', sort=False)

for date, group in grouped:
    # Combine the location (nom) and the logistics event into readable sentences
    texts = [f"In {row['nom']}: {row['logistics_en']}" for _, row in group.iterrows()]
    combined_text = " ".join(texts)

    # Tokenize the combined text
    inputs = tokenizer(combined_text, return_tensors="pt", max_length=1024, truncation=True)
    input_length = inputs["input_ids"].shape[1]

    # Skip days where the combined text is too short to summarize
    if input_length < 15:
        print(f"--- Date: {date} ---")
        print("Note: Activity too low for a summary.")
        print(f"Raw update: {combined_text}\n")
        continue

    # Dynamically adjust generation lengths based on input size to prevent crashes
    gen_max = min(60, input_length + 10)
    gen_min = min(15, input_length // 2)

    # Generate the summary
    summary_ids = model.generate(
        inputs["input_ids"],
        max_length=max(gen_max, 20),
        min_length=max(gen_min, 5),
        do_sample=False
    )

    # Decode back to text
    summary_text = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

    print(f"--- Date: {date} ---")
    print(f"Total Events: {len(group)}")
    print(f"Executive Summary: {summary_text}\n")

print("Processing complete.")

--- Task 2: Daily Events Summarization ---
Loading model: sshleifer/distilbart-cnn-12-6...


Loading weights:   0%|          | 0/358 [00:00<?, ?it/s]

Model loaded successfully.

--- Date: 30-05-2026 ---
Total Events: 1
Executive Summary:  In Bunia: Delivery of 5 tons of medicines (support UG-PDSS). Delivery of demarcation nets to Bunia CME . Delivery of demarcation net to

--- Date: 31-05-2026 ---
Total Events: 1
Executive Summary:  Inauguration and commissioning of the standardised ETC at CME Rwampara, newly rehabilitated .

--- Date: 02-06-2026 ---
Total Events: 18
Executive Summary:  In Rimba: Official handover of 3 vehicles (government donation) by the Governor of Ituri province to CDPS Ituri . Delivery of IPC prevention and control inputs to these health zones . Planned: provision of 2 additional vehicles, installation of Internet kit at Miti-Mur

--- Date: 03-06-2026 ---
Total Events: 3
Executive Summary:  In Mongbwalu: Official handover of 3 vehicles by the Governor to the CDPS Ituri; 2 assigned to Rwampara and Nyankunde HZs. 25 vehicles and 4 ambulances made available. Fuel supply provided for field vehicles and HGR Bunia

-

KeyboardInterrupt: 

### Cell 5: Task 3 - Emotion Analysis
**Goal:** Detect underlying fear, anger, or resistance in the `community_engagement` reports.

In [4]:
print("\n--- Task 3: Emotion Analysis ---")
df_ce = load_data("public_health_response__epidemiological_community_engagement_en__daily.csv")
text_col_ce = "community_engagement_en"

# Initialize Emotion classification pipeline
emotion_pipeline = pipeline("text-classification", model="j-hartmann/emotion-english-distilroberta-base", return_all_scores=False, device=-1)

sample_texts_emo = df_ce[text_col_ce].head(5).tolist()

for i, text in enumerate(sample_texts_emo):
    truncated_text = text[:1500]
    emotion = emotion_pipeline(truncated_text)

    print(f"\n[Text {i+1}]: {text[:200]}...")
    print(f"-> Detected Emotion: {emotion[0]['label'].upper()} (Confidence: {emotion[0]['score']:.2f})")


--- Task 3: Emotion Analysis ---
Downloading: https://raw.githubusercontent.com/INRB-UMIE/BDBV2026-Data/main/data/public_health_response/processed/public_health_response__epidemiological_community_engagement_en__daily.csv


config.json:   0%|          | 0.00/1.00k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  329MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

...

model.safetensors: downloading bytes:           |  0.00B            

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]


[Text 1]: Initial sensitisation of community leaders and religious leaders to counter rumours of 'mystical disease'....
-> Detected Emotion: FEAR (Confidence: 0.93)

[Text 2]: 250 Adventist faithful sensitized in Swahili. 1,636 students (including 990 girls) sensitized in 3 schools. Briefing of RECO on SBC in 5 sites (159 RECO reached)....
-> Detected Emotion: NEUTRAL (Confidence: 0.82)

[Text 3]: 45 taxi-motorcyclists sensitized in 2 parking areas. Delivery of 50 posters to EP Maki Mbudha....
-> Detected Emotion: NEUTRAL (Confidence: 0.89)

[Text 4]: 305 students (including 102 girls) sensitized at EP Kilomoto on preventive measures and alert reporting....
-> Detected Emotion: NEUTRAL (Confidence: 0.90)

[Text 5]: Advocacy with the President of the territorial youth council for his involvement in CREC activities....
-> Detected Emotion: DISGUST (Confidence: 0.67)


### Cell 6: Task 4 - Zero-Shot Text Classification
**Goal:** Categorize generic unstructured text into one of the 9 public health response pillars.

In [5]:
print("\n--- Task 4: Zero-Shot Classification ---")

# Initialize Zero-Shot pipeline
zero_shot_pipeline = pipeline("zero-shot-classification", model="valhalla/distilbart-mnli-12-3", device=-1)

# Define our candidate categories based on the 9 pillars
candidate_labels = [
    "community engagement",
    "coordination",
    "infection prevention and control",
    "laboratory",
    "logistics",
    "case management",
    "monitoring and surveillance",
    "protection from sexual exploitation and abuse",
    "security"
]

# Simulate a generic, unclassified incoming report
generic_report = (
    "A team of doctors was dispatched to the newly rehabilitated ETC in Rwampara to handle the influx of confirmed patients. "
    "However, armed groups on the roads delayed the delivery of essential PPE and medical supplies."
)

print(f"[Incoming Report]: {generic_report}")

# Classify the text
classification_result = zero_shot_pipeline(generic_report, candidate_labels, multi_label=True)

print("\n-> Top 3 Predicted Pillars:")
for i in range(3):
    print(f"   {i+1}. {classification_result['labels'][i]} (Score: {classification_result['scores'][i]:.2f})")


--- Task 4: Zero-Shot Classification ---


config.json:   0%|          | 0.00/1.39k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.02GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/283 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

[Incoming Report]: A team of doctors was dispatched to the newly rehabilitated ETC in Rwampara to handle the influx of confirmed patients. However, armed groups on the roads delayed the delivery of essential PPE and medical supplies.

-> Top 3 Predicted Pillars:
   1. case management (Score: 0.83)
   2. logistics (Score: 0.81)
   3. coordination (Score: 0.73)
